In [1]:
# ============================================================
# NOTEBOOK 02: FEATURE ENGINEERING
# Proyecto: Credit Risk Scoring ML
# Autor: Marín Serrato Barrios
# Descripción: Construcción de variables predictivas
#              para el modelo de scoring crediticio.
#              Basado en hallazgos del EDA (notebook 01).
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import os

# Configuración
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

# Ruta base del proyecto (ruta absoluta, siempre funciona)
RUTA_PROYECTO = "c:/Users/Marin/Documents/PROYECTO ML_OPS/credit-risk-scoring-ml"
os.chdir(RUTA_PROYECTO)

# Ruta al dataset
RUTA_ZIP = "data/raw/home-credit-default-risk.zip"

print(f"Directorio de trabajo: {os.getcwd()}")
print(f"Dataset disponible:    {os.path.exists(RUTA_ZIP)}")
print("\n✅ Configuración correcta")

Directorio de trabajo: c:\Users\Marin\Documents\PROYECTO ML_OPS\credit-risk-scoring-ml
Dataset disponible:    True

✅ Configuración correcta


In [2]:
# --- CARGA DEL DATASET ---

# Cargamos application_train.csv igual que en el EDA
with zipfile.ZipFile(RUTA_ZIP, 'r') as z:
    with z.open('application_train.csv') as f:
        df = pd.read_csv(f)

print(f"Dataset cargado: {df.shape[0]:,} filas, {df.shape[1]:,} columnas")

# Separar TARGET y SK_ID_CURR del resto
# Los guardamos aparte para no mezclarlos con los features
TARGET = df['TARGET'].copy()
ID     = df['SK_ID_CURR'].copy()

print(f"TARGET separado:  {TARGET.shape[0]:,} valores")
print(f"Tasa de mora:     {TARGET.mean()*100:.2f}%")

Dataset cargado: 307,511 filas, 122 columnas
TARGET separado:  307,511 valores
Tasa de mora:     8.07%


In [3]:
# --- PASO 1: ELIMINAR VARIABLES DE BAJO VALOR ---

# Del EDA identificamos tres grupos a descartar:
#
# GRUPO 1: Variables de características del edificio
#          >50% de nulos y bajo poder predictivo
#
# GRUPO 2: FLAG_DOCUMENT con correlación casi cero
#
# GRUPO 3: Variables redundantes o de control

# GRUPO 1: Características del edificio
# Son todas las que terminan en _AVG, _MEDI, _MODE
# más algunas variables específicas del inmueble
cols_edificio = [col for col in df.columns
                 if col.endswith('_AVG')
                 or col.endswith('_MEDI')
                 or col.endswith('_MODE')]

print(f"Variables de edificio a eliminar: {len(cols_edificio)}")

# GRUPO 2: FLAG_DOCUMENT con correlación casi cero
# Del EDA: FLAG_DOCUMENT_5/7/10/12/19/20 ≈ 0.0002
# Por consistencia eliminamos todos excepto FLAG_DOCUMENT_3
# que sí tenía correlación relevante (0.0443)
cols_documentos_bajos = [
    'FLAG_DOCUMENT_2',  'FLAG_DOCUMENT_4',  'FLAG_DOCUMENT_5',
    'FLAG_DOCUMENT_6',  'FLAG_DOCUMENT_7',  'FLAG_DOCUMENT_8',
    'FLAG_DOCUMENT_9',  'FLAG_DOCUMENT_10', 'FLAG_DOCUMENT_11',
    'FLAG_DOCUMENT_12', 'FLAG_DOCUMENT_13', 'FLAG_DOCUMENT_14',
    'FLAG_DOCUMENT_15', 'FLAG_DOCUMENT_16', 'FLAG_DOCUMENT_17',
    'FLAG_DOCUMENT_18', 'FLAG_DOCUMENT_19', 'FLAG_DOCUMENT_20',
    'FLAG_DOCUMENT_21'
]

print(f"FLAG_DOCUMENT a eliminar:         {len(cols_documentos_bajos)}")

# GRUPO 3: Variables de consultas al Buró de muy baja frecuencia
# Hora y semana tienen correlación ≈ 0.0008
cols_buro_bajas = [
    'AMT_REQ_CREDIT_BUREAU_HOUR',
    'AMT_REQ_CREDIT_BUREAU_WEEK',
]

print(f"Consultas Buró a eliminar:        {len(cols_buro_bajas)}")

# Eliminar todas
cols_eliminar = cols_edificio + cols_documentos_bajos + cols_buro_bajas
df.drop(columns=cols_eliminar, inplace=True)

print(f"\nColumnas antes: 122")
print(f"Columnas después: {df.shape[1]}")
print(f"Variables eliminadas: {len(cols_eliminar)}")

Variables de edificio a eliminar: 47
FLAG_DOCUMENT a eliminar:         19
Consultas Buró a eliminar:        2

Columnas antes: 122
Columnas después: 54
Variables eliminadas: 68


In [4]:
# --- PASO 2 COMPLETO CORREGIDO: TRATAMIENTO DE VALORES FALTANTES ---
# Sintaxis correcta para pandas 3.0 (Copy-on-Write)
# Usar: df['col'] = df['col'].fillna(valor)
# NO usar: df['col'].fillna(valor, inplace=True)

print("ESTRATEGIA DE TRATAMIENTO DE MISSINGS:")
print("="*60)
print("LightGBM:   NaN preservados + FLAGS de ausencia")
print("Scorecard:  Imputación completa + FLAGS de ausencia")
print()

# Recrear los dos dataframes desde df limpio
# (el df que tiene 54 columnas después del paso 1)
df_lgbm      = df.copy()
df_scorecard = df.copy()

# ── PASO 2A: Crear FLAGS de ausencia (aplica a ambos) ─────
print("PASO 2A: Creando FLAGS de ausencia...")
print("-"*40)

# EXT_SOURCE_1: 56.4% nulos
df_lgbm['FLAG_EXT_SOURCE_1_NULO'] = (
    df_lgbm['EXT_SOURCE_1'].isna().astype(int))
df_scorecard['FLAG_EXT_SOURCE_1_NULO'] = (
    df_scorecard['EXT_SOURCE_1'].isna().astype(int))
print(f"FLAG_EXT_SOURCE_1_NULO: "
      f"{df_lgbm['FLAG_EXT_SOURCE_1_NULO'].sum():,} casos "
      f"({df_lgbm['FLAG_EXT_SOURCE_1_NULO'].mean()*100:.1f}%)")

# EXT_SOURCE_3: 19.8% nulos
df_lgbm['FLAG_EXT_SOURCE_3_NULO'] = (
    df_lgbm['EXT_SOURCE_3'].isna().astype(int))
df_scorecard['FLAG_EXT_SOURCE_3_NULO'] = (
    df_scorecard['EXT_SOURCE_3'].isna().astype(int))
print(f"FLAG_EXT_SOURCE_3_NULO: "
      f"{df_lgbm['FLAG_EXT_SOURCE_3_NULO'].sum():,} casos "
      f"({df_lgbm['FLAG_EXT_SOURCE_3_NULO'].mean()*100:.1f}%)")

# OCCUPATION_TYPE: 31.3% nulos
df_lgbm['FLAG_OCCUPATION_NULO'] = (
    df_lgbm['OCCUPATION_TYPE'].isna().astype(int))
df_scorecard['FLAG_OCCUPATION_NULO'] = (
    df_scorecard['OCCUPATION_TYPE'].isna().astype(int))
print(f"FLAG_OCCUPATION_NULO:   "
      f"{df_lgbm['FLAG_OCCUPATION_NULO'].sum():,} casos "
      f"({df_lgbm['FLAG_OCCUPATION_NULO'].mean()*100:.1f}%)")

# ── PASO 2B: Imputación SOLO para df_scorecard ────────────
print()
print("PASO 2B: Imputación para df_scorecard...")
print("-"*40)

# Calcular medianas ANTES de imputar
mediana_ext1   = df_scorecard['EXT_SOURCE_1'].median()
mediana_ext2   = df_scorecard['EXT_SOURCE_2'].median()
mediana_ext3   = df_scorecard['EXT_SOURCE_3'].median()
moda_suite     = df_scorecard['NAME_TYPE_SUITE'].mode()[0]

# EXT_SOURCE_1
df_scorecard['EXT_SOURCE_1'] = (
    df_scorecard['EXT_SOURCE_1'].fillna(mediana_ext1))
print(f"EXT_SOURCE_1    → mediana: {mediana_ext1:.4f}")

# EXT_SOURCE_2
df_scorecard['EXT_SOURCE_2'] = (
    df_scorecard['EXT_SOURCE_2'].fillna(mediana_ext2))
print(f"EXT_SOURCE_2    → mediana: {mediana_ext2:.4f}")

# EXT_SOURCE_3
df_scorecard['EXT_SOURCE_3'] = (
    df_scorecard['EXT_SOURCE_3'].fillna(mediana_ext3))
print(f"EXT_SOURCE_3    → mediana: {mediana_ext3:.4f}")

# OCCUPATION_TYPE: categoría propia 'Unknown'
df_scorecard['OCCUPATION_TYPE'] = (
    df_scorecard['OCCUPATION_TYPE'].fillna('Unknown'))
print(f"OCCUPATION_TYPE → 'Unknown'")

# NAME_TYPE_SUITE: moda
df_scorecard['NAME_TYPE_SUITE'] = (
    df_scorecard['NAME_TYPE_SUITE'].fillna(moda_suite))
print(f"NAME_TYPE_SUITE → moda: '{moda_suite}'")

# Variables numéricas con pocos nulos
cols_mediana = ['AMT_ANNUITY', 'AMT_GOODS_PRICE', 'CNT_FAM_MEMBERS']
for col in cols_mediana:
    nulos = df_scorecard[col].isna().sum()
    if nulos > 0:
        mediana = df_scorecard[col].median()
        df_scorecard[col] = df_scorecard[col].fillna(mediana)
        print(f"{col:<20} → mediana: {mediana:.0f} ({nulos} casos)")

# ── PASO 2C: Imputación mínima para df_lgbm ───────────────
# LightGBM maneja NaN nativamente en EXT_SOURCE y OCCUPATION
# Solo imputamos variables con muy pocos nulos donde
# el NaN no aporta información
print()
print("PASO 2C: Imputación mínima para df_lgbm...")
print("-"*40)

for col in cols_mediana:
    nulos = df_lgbm[col].isna().sum()
    if nulos > 0:
        mediana = df_lgbm[col].median()
        df_lgbm[col] = df_lgbm[col].fillna(mediana)
        print(f"{col:<20} → mediana: {mediana:.0f} ({nulos} casos)")

# ── VERIFICACIÓN FINAL ────────────────────────────────────
print()
print("VERIFICACIÓN FINAL:")
print("="*60)

nulos_lgbm  = df_lgbm.isnull().sum()
nulos_score = df_scorecard.isnull().sum()

vars_nan_lgbm  = nulos_lgbm[nulos_lgbm > 0]
vars_nan_score = nulos_score[nulos_score > 0]

print(f"df_lgbm      - variables con NaN: {len(vars_nan_lgbm)}")
if len(vars_nan_lgbm) > 0:
    for col, n in vars_nan_lgbm.items():
        print(f"  {col}: {n:,} NaN  ← LightGBM los maneja nativamente")

print()
print(f"df_scorecard - variables con NaN: {len(vars_nan_score)} ← debe ser 0")
if len(vars_nan_score) > 0:
    print("  PROBLEMA: aún hay NaN en scorecard")
    print(vars_nan_score.to_string())
else:
    print("  ✅ Sin NaN, listo para regresión logística")

print()
print(f"Shape df_lgbm:      {df_lgbm.shape}")
print(f"Shape df_scorecard: {df_scorecard.shape}")

ESTRATEGIA DE TRATAMIENTO DE MISSINGS:
LightGBM:   NaN preservados + FLAGS de ausencia
Scorecard:  Imputación completa + FLAGS de ausencia

PASO 2A: Creando FLAGS de ausencia...
----------------------------------------
FLAG_EXT_SOURCE_1_NULO: 173,378 casos (56.4%)
FLAG_EXT_SOURCE_3_NULO: 60,965 casos (19.8%)
FLAG_OCCUPATION_NULO:   96,391 casos (31.3%)

PASO 2B: Imputación para df_scorecard...
----------------------------------------
EXT_SOURCE_1    → mediana: 0.5060
EXT_SOURCE_2    → mediana: 0.5660
EXT_SOURCE_3    → mediana: 0.5353
OCCUPATION_TYPE → 'Unknown'
NAME_TYPE_SUITE → moda: 'Unaccompanied'
AMT_ANNUITY          → mediana: 24903 (12 casos)
AMT_GOODS_PRICE      → mediana: 450000 (278 casos)
CNT_FAM_MEMBERS      → mediana: 2 (2 casos)

PASO 2C: Imputación mínima para df_lgbm...
----------------------------------------
AMT_ANNUITY          → mediana: 24903 (12 casos)
AMT_GOODS_PRICE      → mediana: 450000 (278 casos)
CNT_FAM_MEMBERS      → mediana: 2 (2 casos)

VERIFICACIÓN FINAL

In [5]:
# --- PASO 2D: Imputación de variables pendientes en df_scorecard ---

print("PASO 2D: Imputando variables pendientes en df_scorecard...")
print("-"*40)

# OWN_CAR_AGE: 202,929 nulos (65.9%)
# Los nulos son clientes SIN auto (FLAG_OWN_CAR = 'N')
# No tiene sentido imputar con mediana de los que SÍ tienen auto
# Imputamos con 0 → "no tiene auto, antigüedad = 0"
df_scorecard['OWN_CAR_AGE'] = df_scorecard['OWN_CAR_AGE'].fillna(0)
print(f"OWN_CAR_AGE              → 0 (sin auto)")

# Variables del círculo social
# OBS/DEF_30/60_CNT_SOCIAL_CIRCLE: 1,021 nulos
# Observaciones y defaults en el círculo social del cliente
# Imputamos con 0 → sin observaciones registradas
cols_social = ['OBS_30_CNT_SOCIAL_CIRCLE', 'DEF_30_CNT_SOCIAL_CIRCLE',
               'OBS_60_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE']
for col in cols_social:
    df_scorecard[col] = df_scorecard[col].fillna(0)
    print(f"{col:<30} → 0")

# DAYS_LAST_PHONE_CHANGE: solo 1 nulo
# Imputamos con mediana
mediana_phone = df_scorecard['DAYS_LAST_PHONE_CHANGE'].median()
df_scorecard['DAYS_LAST_PHONE_CHANGE'] = (
    df_scorecard['DAYS_LAST_PHONE_CHANGE'].fillna(mediana_phone))
print(f"DAYS_LAST_PHONE_CHANGE   → mediana: {mediana_phone:.0f}")

# AMT_REQ_CREDIT_BUREAU_DAY/MON/QRT/YEAR: 41,519 nulos
# Son consultas al buró en distintos periodos
# Nulos probablemente = sin consultas recientes
# Imputamos con 0 → sin consultas
cols_buro = ['AMT_REQ_CREDIT_BUREAU_DAY',
             'AMT_REQ_CREDIT_BUREAU_MON',
             'AMT_REQ_CREDIT_BUREAU_QRT',
             'AMT_REQ_CREDIT_BUREAU_YEAR']
for col in cols_buro:
    df_scorecard[col] = df_scorecard[col].fillna(0)
    print(f"{col:<30} → 0")

# VERIFICACIÓN FINAL
print()
print("VERIFICACIÓN FINAL COMPLETA:")
print("="*60)
nulos_score = df_scorecard.isnull().sum()
vars_nan_score = nulos_score[nulos_score > 0]

print(f"df_scorecard - variables con NaN: {len(vars_nan_score)}", end=" ")
if len(vars_nan_score) == 0:
    print("✅ LISTO")
else:
    print("❌ PROBLEMA")
    print(vars_nan_score.to_string())

print(f"\ndf_lgbm      - variables con NaN: "
      f"{(df_lgbm.isnull().sum() > 0).sum()} "
      f"(manejados nativamente por LightGBM) ✅")

print(f"\nShape df_lgbm:      {df_lgbm.shape}")
print(f"Shape df_scorecard: {df_scorecard.shape}")

PASO 2D: Imputando variables pendientes en df_scorecard...
----------------------------------------
OWN_CAR_AGE              → 0 (sin auto)
OBS_30_CNT_SOCIAL_CIRCLE       → 0
DEF_30_CNT_SOCIAL_CIRCLE       → 0
OBS_60_CNT_SOCIAL_CIRCLE       → 0
DEF_60_CNT_SOCIAL_CIRCLE       → 0
DAYS_LAST_PHONE_CHANGE   → mediana: -757
AMT_REQ_CREDIT_BUREAU_DAY      → 0
AMT_REQ_CREDIT_BUREAU_MON      → 0
AMT_REQ_CREDIT_BUREAU_QRT      → 0
AMT_REQ_CREDIT_BUREAU_YEAR     → 0

VERIFICACIÓN FINAL COMPLETA:
df_scorecard - variables con NaN: 0 ✅ LISTO

df_lgbm      - variables con NaN: 15 (manejados nativamente por LightGBM) ✅

Shape df_lgbm:      (307511, 57)
Shape df_scorecard: (307511, 57)


In [6]:
# --- PASO 3: CONSTRUCCIÓN DE FEATURES DERIVADOS ---

# Aquí transformamos variables crudas en variables
# con significado de negocio directo.
# Estos features capturan relaciones que el modelo
# no podría descubrir fácilmente por sí solo.

print("PASO 3: CONSTRUCCIÓN DE FEATURES DERIVADOS")
print("="*60)

# ── FEATURE 1: DTI (Debt-to-Income Ratio) ─────────────────
# Vimos en el EDA que AMT_INCOME_TOTAL es MENSUAL
# AMT_ANNUITY también es mensual
# DTI = pago mensual / ingreso mensual

for dataset in [df_lgbm, df_scorecard]:
    dataset['DTI'] = dataset['AMT_ANNUITY'] / dataset['AMT_INCOME_TOTAL']

print(f"DTI:")
print(f"  Media:   {df_lgbm['DTI'].mean():.4f} ({df_lgbm['DTI'].mean()*100:.1f}%)")
print(f"  Mediana: {df_lgbm['DTI'].median():.4f} ({df_lgbm['DTI'].median()*100:.1f}%)")

# ── FEATURE 2: RATIO CRÉDITO / INGRESO ANUAL ──────────────
# Cuántos meses de ingreso representa el crédito total
# Mide el tamaño relativo del crédito vs capacidad de pago

for dataset in [df_lgbm, df_scorecard]:
    dataset['RATIO_CREDITO_INGRESO'] = (
        dataset['AMT_CREDIT'] / dataset['AMT_INCOME_TOTAL'])

print(f"\nRATIO_CREDITO_INGRESO:")
print(f"  Media:   {df_lgbm['RATIO_CREDITO_INGRESO'].mean():.2f} meses de ingreso")
print(f"  Mediana: {df_lgbm['RATIO_CREDITO_INGRESO'].median():.2f} meses de ingreso")

# ── FEATURE 3: PLAZO ESTIMADO DEL CRÉDITO ─────────────────
# AMT_CREDIT / AMT_ANNUITY = número de meses del crédito
# Vimos en el EDA que esto da ~20 meses en promedio

for dataset in [df_lgbm, df_scorecard]:
    dataset['PLAZO_MESES'] = (
        dataset['AMT_CREDIT'] / dataset['AMT_ANNUITY'])

print(f"\nPLAZO_MESES:")
print(f"  Media:   {df_lgbm['PLAZO_MESES'].mean():.1f} meses")
print(f"  Mediana: {df_lgbm['PLAZO_MESES'].median():.1f} meses")

# ── FEATURE 4: EDAD EN AÑOS ────────────────────────────────
# Ya lo calculamos en el EDA, lo incluimos formalmente

for dataset in [df_lgbm, df_scorecard]:
    dataset['EDAD_AÑOS'] = (-dataset['DAYS_BIRTH'] / 365).astype(int)

print(f"\nEDAD_AÑOS:")
print(f"  Media:   {df_lgbm['EDAD_AÑOS'].mean():.1f} años")
print(f"  Mediana: {df_lgbm['EDAD_AÑOS'].median():.1f} años")

# ── FEATURE 5: ANTIGÜEDAD LABORAL ─────────────────────────
# Tratar valor centinela 365243 (pensionados/desempleados)
# y convertir a años positivos

for dataset in [df_lgbm, df_scorecard]:
    # Reemplazar centinela con NaN
    days_employed_limpio = dataset['DAYS_EMPLOYED'].replace(365243, np.nan)
    # Convertir a años positivos
    dataset['ANTIGUEDAD_LABORAL_AÑOS'] = (-days_employed_limpio / 365)
    # Flag de pensionado/desempleado
    dataset['FLAG_PENSIONADO_DESEMPLEADO'] = (
        dataset['DAYS_EMPLOYED'] == 365243).astype(int)

# Para scorecard: imputar NaN de antigüedad con 0
df_scorecard['ANTIGUEDAD_LABORAL_AÑOS'] = (
    df_scorecard['ANTIGUEDAD_LABORAL_AÑOS'].fillna(0))

print(f"\nANTIGUEDAD_LABORAL_AÑOS:")
print(f"  Media:   {df_lgbm['ANTIGUEDAD_LABORAL_AÑOS'].mean():.1f} años")
print(f"  Mediana: {df_lgbm['ANTIGUEDAD_LABORAL_AÑOS'].median():.1f} años")
print(f"  Pensionados/desempleados: "
      f"{df_lgbm['FLAG_PENSIONADO_DESEMPLEADO'].sum():,}")

# ── FEATURE 6: EXT_SOURCE PROMEDIO ────────────────────────
# Combinar los tres scores externos en un promedio
# Captura el perfil crediticio general del cliente

for dataset in [df_lgbm, df_scorecard]:
    dataset['EXT_SOURCE_PROMEDIO'] = (
        dataset[['EXT_SOURCE_1',
                 'EXT_SOURCE_2',
                 'EXT_SOURCE_3']].mean(axis=1))

print(f"\nEXT_SOURCE_PROMEDIO:")
print(f"  Media:   {df_lgbm['EXT_SOURCE_PROMEDIO'].mean():.4f}")
print(f"  Mediana: {df_lgbm['EXT_SOURCE_PROMEDIO'].median():.4f}")

# Diferencia buenos vs malos
media_buenos = df_lgbm[df_lgbm['TARGET']==0]['EXT_SOURCE_PROMEDIO'].mean()
media_malos  = df_lgbm[df_lgbm['TARGET']==1]['EXT_SOURCE_PROMEDIO'].mean()
print(f"  Buenos:  {media_buenos:.4f}")
print(f"  Malos:   {media_malos:.4f}")
print(f"  Diferencia: {media_buenos-media_malos:.4f}")

# ── FEATURE 7: EXT_SOURCE MÍNIMO ──────────────────────────
# El score más bajo de los tres
# Captura el peor antecedente crediticio

for dataset in [df_lgbm, df_scorecard]:
    dataset['EXT_SOURCE_MIN'] = (
        dataset[['EXT_SOURCE_1',
                 'EXT_SOURCE_2',
                 'EXT_SOURCE_3']].min(axis=1))

print(f"\nEXT_SOURCE_MIN (peor score):")
media_buenos = df_lgbm[df_lgbm['TARGET']==0]['EXT_SOURCE_MIN'].mean()
media_malos  = df_lgbm[df_lgbm['TARGET']==1]['EXT_SOURCE_MIN'].mean()
print(f"  Buenos:     {media_buenos:.4f}")
print(f"  Malos:      {media_malos:.4f}")
print(f"  Diferencia: {media_buenos-media_malos:.4f}")

# ── RESUMEN ────────────────────────────────────────────────
print()
print("FEATURES DERIVADOS CREADOS:")
print("="*60)
features_nuevos = ['DTI', 'RATIO_CREDITO_INGRESO', 'PLAZO_MESES',
                   'EDAD_AÑOS', 'ANTIGUEDAD_LABORAL_AÑOS',
                   'FLAG_PENSIONADO_DESEMPLEADO',
                   'EXT_SOURCE_PROMEDIO', 'EXT_SOURCE_MIN']
for f in features_nuevos:
    print(f"  ✅ {f}")

print(f"\nShape df_lgbm:      {df_lgbm.shape}")
print(f"Shape df_scorecard: {df_scorecard.shape}")

PASO 3: CONSTRUCCIÓN DE FEATURES DERIVADOS
DTI:
  Media:   0.1809 (18.1%)
  Mediana: 0.1628 (16.3%)

RATIO_CREDITO_INGRESO:
  Media:   3.96 meses de ingreso
  Mediana: 3.27 meses de ingreso

PLAZO_MESES:
  Media:   21.6 meses
  Mediana: 20.0 meses

EDAD_AÑOS:
  Media:   43.4 años
  Mediana: 43.0 años

ANTIGUEDAD_LABORAL_AÑOS:
  Media:   6.5 años
  Mediana: 4.5 años
  Pensionados/desempleados: 55,374

EXT_SOURCE_PROMEDIO:
  Media:   0.5093
  Mediana: 0.5245
  Buenos:  0.5191
  Malos:   0.3970
  Diferencia: 0.1221

EXT_SOURCE_MIN (peor score):
  Buenos:     0.4099
  Malos:      0.2824
  Diferencia: 0.1275

FEATURES DERIVADOS CREADOS:
  ✅ DTI
  ✅ RATIO_CREDITO_INGRESO
  ✅ PLAZO_MESES
  ✅ EDAD_AÑOS
  ✅ ANTIGUEDAD_LABORAL_AÑOS
  ✅ FLAG_PENSIONADO_DESEMPLEADO
  ✅ EXT_SOURCE_PROMEDIO
  ✅ EXT_SOURCE_MIN

Shape df_lgbm:      (307511, 65)
Shape df_scorecard: (307511, 65)


In [7]:
# --- PASO 4: VARIABLES DE INTERACCIÓN ---

# Las variables de interacción combinan dos variables
# para capturar efectos conjuntos.
#
# Ejemplo: Un DTI alto es más peligroso en un cliente
# joven que en uno mayor con historial limpio.
# La interacción DTI * EDAD captura eso.

print("PASO 4: VARIABLES DE INTERACCIÓN")
print("="*60)

# ── INTERACCIÓN 1: DTI × EXT_SOURCE_2 ─────────────────────
# Cliente con alto DTI Y bajo score externo
# Es el perfil de mayor riesgo combinado
# Valores altos = mayor riesgo

for dataset in [df_lgbm, df_scorecard]:
    dataset['DTI_X_EXT2'] = (
        dataset['DTI'] * (1 - dataset['EXT_SOURCE_2']))

media_buenos = df_lgbm[df_lgbm['TARGET']==0]['DTI_X_EXT2'].mean()
media_malos  = df_lgbm[df_lgbm['TARGET']==1]['DTI_X_EXT2'].mean()
print(f"DTI_X_EXT2 (DTI × riesgo EXT_SOURCE_2):")
print(f"  Buenos: {media_buenos:.4f}  Malos: {media_malos:.4f}  "
      f"Diferencia: {media_malos-media_buenos:.4f}")

# ── INTERACCIÓN 2: EDAD × EXT_SOURCE_PROMEDIO ─────────────
# Cliente joven con bajo score = perfil muy riesgoso
# Normalizamos edad a escala 0-1 para que sea comparable

edad_norm = (df_lgbm['EDAD_AÑOS'] - df_lgbm['EDAD_AÑOS'].min()) / \
            (df_lgbm['EDAD_AÑOS'].max() - df_lgbm['EDAD_AÑOS'].min())

for dataset in [df_lgbm, df_scorecard]:
    edad_norm_d = ((dataset['EDAD_AÑOS'] - dataset['EDAD_AÑOS'].min()) /
                   (dataset['EDAD_AÑOS'].max() - dataset['EDAD_AÑOS'].min()))
    # Mayor valor = más joven Y menor score = más riesgo
    dataset['RIESGO_EDAD_SCORE'] = (
        (1 - edad_norm_d) * (1 - dataset['EXT_SOURCE_PROMEDIO']))

media_buenos = df_lgbm[df_lgbm['TARGET']==0]['RIESGO_EDAD_SCORE'].mean()
media_malos  = df_lgbm[df_lgbm['TARGET']==1]['RIESGO_EDAD_SCORE'].mean()
print(f"\nRIESGO_EDAD_SCORE (juventud × bajo score):")
print(f"  Buenos: {media_buenos:.4f}  Malos: {media_malos:.4f}  "
      f"Diferencia: {media_malos-media_buenos:.4f}")

# ── INTERACCIÓN 3: ANTIGÜEDAD × DTI ───────────────────────
# Poca antigüedad laboral Y alto DTI = doble señal de riesgo

for dataset in [df_lgbm, df_scorecard]:
    antiguedad_segura = dataset['ANTIGUEDAD_LABORAL_AÑOS'].fillna(0)
    dataset['DTI_X_ANTIGUEDAD'] = dataset['DTI'] / (antiguedad_segura + 1)
    # +1 para evitar división por cero

media_buenos = df_lgbm[df_lgbm['TARGET']==0]['DTI_X_ANTIGUEDAD'].mean()
media_malos  = df_lgbm[df_lgbm['TARGET']==1]['DTI_X_ANTIGUEDAD'].mean()
print(f"\nDTI_X_ANTIGUEDAD (DTI / antigüedad+1):")
print(f"  Buenos: {media_buenos:.4f}  Malos: {media_malos:.4f}  "
      f"Diferencia: {media_malos-media_buenos:.4f}")

# ── INTERACCIÓN 4: CONSULTAS BURÓ RECIENTES ───────────────
# Total de consultas en el último año como señal de estrés
# (ya vimos que las consultas al Buró son señal de riesgo)

for dataset in [df_lgbm, df_scorecard]:
    dataset['CONSULTAS_BURO_TOTAL'] = (
        dataset['AMT_REQ_CREDIT_BUREAU_DAY'].fillna(0) +
        dataset['AMT_REQ_CREDIT_BUREAU_MON'].fillna(0) +
        dataset['AMT_REQ_CREDIT_BUREAU_QRT'].fillna(0) +
        dataset['AMT_REQ_CREDIT_BUREAU_YEAR'].fillna(0)
    )

media_buenos = df_lgbm[df_lgbm['TARGET']==0]['CONSULTAS_BURO_TOTAL'].mean()
media_malos  = df_lgbm[df_lgbm['TARGET']==1]['CONSULTAS_BURO_TOTAL'].mean()
print(f"\nCONSULTAS_BURO_TOTAL (suma consultas al año):")
print(f"  Buenos: {media_buenos:.2f}  Malos: {media_malos:.2f}  "
      f"Diferencia: {media_malos-media_buenos:.2f}")

# ── RESUMEN ────────────────────────────────────────────────
print()
print("VARIABLES DE INTERACCIÓN CREADAS:")
print("="*60)
interacciones = ['DTI_X_EXT2', 'RIESGO_EDAD_SCORE',
                 'DTI_X_ANTIGUEDAD', 'CONSULTAS_BURO_TOTAL']
for f in interacciones:
    print(f"  ✅ {f}")

print(f"\nShape df_lgbm:      {df_lgbm.shape}")
print(f"Shape df_scorecard: {df_scorecard.shape}")

PASO 4: VARIABLES DE INTERACCIÓN
DTI_X_EXT2 (DTI × riesgo EXT_SOURCE_2):
  Buenos: 0.0866  Malos: 0.1098  Diferencia: 0.0232

RIESGO_EDAD_SCORE (juventud × bajo score):
  Buenos: 0.2577  Malos: 0.3630  Diferencia: 0.1052

DTI_X_ANTIGUEDAD (DTI / antigüedad+1):
  Buenos: 0.0702  Malos: 0.0691  Diferencia: -0.0011

CONSULTAS_BURO_TOTAL (suma consultas al año):
  Buenos: 2.11  Malos: 2.09  Diferencia: -0.02

VARIABLES DE INTERACCIÓN CREADAS:
  ✅ DTI_X_EXT2
  ✅ RIESGO_EDAD_SCORE
  ✅ DTI_X_ANTIGUEDAD
  ✅ CONSULTAS_BURO_TOTAL

Shape df_lgbm:      (307511, 69)
Shape df_scorecard: (307511, 69)


In [8]:
# --- LIMPIEZA: ELIMINAR INTERACCIONES SIN PODER PREDICTIVO ---

# DTI_X_ANTIGUEDAD y CONSULTAS_BURO_TOTAL no discriminan
# entre buenos y malos (diferencia < 0.01)
# Incluirlos sería agregar ruido al modelo

cols_descartar = ['DTI_X_ANTIGUEDAD', 'CONSULTAS_BURO_TOTAL']

for dataset in [df_lgbm, df_scorecard]:
    dataset.drop(columns=cols_descartar, inplace=True)

print("INTERACCIONES DESCARTADAS POR BAJO PODER PREDICTIVO:")
for col in cols_descartar:
    print(f"  ❌ {col}")

print()
print("INTERACCIONES CONSERVADAS:")
print(f"  ✅ DTI_X_EXT2          diferencia: 0.0232")
print(f"  ✅ RIESGO_EDAD_SCORE   diferencia: 0.1052")

print(f"\nShape df_lgbm:      {df_lgbm.shape}")
print(f"Shape df_scorecard: {df_scorecard.shape}")

INTERACCIONES DESCARTADAS POR BAJO PODER PREDICTIVO:
  ❌ DTI_X_ANTIGUEDAD
  ❌ CONSULTAS_BURO_TOTAL

INTERACCIONES CONSERVADAS:
  ✅ DTI_X_EXT2          diferencia: 0.0232
  ✅ RIESGO_EDAD_SCORE   diferencia: 0.1052

Shape df_lgbm:      (307511, 67)
Shape df_scorecard: (307511, 67)


In [9]:
# Verificación rápida del estado actual
print(f"df_lgbm shape:      {df_lgbm.shape}")
print(f"df_scorecard shape: {df_scorecard.shape}")
print(f"TARGET shape:       {TARGET.shape}")

df_lgbm shape:      (307511, 67)
df_scorecard shape: (307511, 67)
TARGET shape:       (307511,)


In [10]:
# --- PASO 5: ENCODING DE VARIABLES CATEGÓRICAS ---

# Los modelos de ML necesitan números, no texto.
# Tenemos dos estrategias según el modelo:
#
# PARA LIGHTGBM:
#   Label Encoding simple (0, 1, 2, 3...)
#   LightGBM puede manejar variables categóricas
#   numéricas directamente
#
# PARA SCORECARD LOGÍSTICO:
#   One-Hot Encoding
#   La regresión logística necesita variables binarias

from sklearn.preprocessing import LabelEncoder

print("PASO 5: ENCODING DE VARIABLES CATEGÓRICAS")
print("="*60)

# Identificar variables categóricas
vars_cat_lgbm  = df_lgbm.select_dtypes(include=['object']).columns.tolist()
vars_cat_score = df_scorecard.select_dtypes(include=['object']).columns.tolist()

print(f"Variables categóricas encontradas: {len(vars_cat_lgbm)}")
for v in vars_cat_lgbm:
    print(f"  {v}: {df_lgbm[v].nunique()} categorías únicas")

# ── LABEL ENCODING para df_lgbm ───────────────────────────
print()
print("LABEL ENCODING para df_lgbm:")
print("-"*40)

mapeo_categorias = {}
le = LabelEncoder()

for col in vars_cat_lgbm:
    df_lgbm[col] = df_lgbm[col].fillna('Unknown')
    df_lgbm[col] = le.fit_transform(df_lgbm[col])
    mapeo_categorias[col] = dict(
        zip(le.classes_, le.transform(le.classes_)))
    print(f"  {col}: {len(mapeo_categorias[col])} categorías → números")

# ── ONE-HOT ENCODING para df_scorecard ────────────────────
print()
print("ONE-HOT ENCODING para df_scorecard:")
print("-"*40)

df_scorecard = pd.get_dummies(
    df_scorecard,
    columns=vars_cat_score,
    drop_first=True,   # evita multicolinealidad
    dtype=int          # 0 y 1 en lugar de True/False
)

print(f"Columnas antes:  {df_lgbm.shape[1]}")
print(f"Columnas después: {df_scorecard.shape[1]}")
print(f"Columnas nuevas: {df_scorecard.shape[1] - df_lgbm.shape[1]}")

# ── VERIFICACIÓN ──────────────────────────────────────────
print()
print("VERIFICACIÓN:")
print(f"  df_lgbm categóricas restantes:      "
      f"{df_lgbm.select_dtypes(include=['object']).shape[1]}")
print(f"  df_scorecard categóricas restantes: "
      f"{df_scorecard.select_dtypes(include=['object']).shape[1]}")
print(f"\n  df_lgbm shape:      {df_lgbm.shape}")
print(f"  df_scorecard shape: {df_scorecard.shape}")

PASO 5: ENCODING DE VARIABLES CATEGÓRICAS
Variables categóricas encontradas: 12
  NAME_CONTRACT_TYPE: 2 categorías únicas
  CODE_GENDER: 3 categorías únicas
  FLAG_OWN_CAR: 2 categorías únicas
  FLAG_OWN_REALTY: 2 categorías únicas
  NAME_TYPE_SUITE: 7 categorías únicas
  NAME_INCOME_TYPE: 8 categorías únicas
  NAME_EDUCATION_TYPE: 5 categorías únicas
  NAME_FAMILY_STATUS: 6 categorías únicas
  NAME_HOUSING_TYPE: 6 categorías únicas
  OCCUPATION_TYPE: 18 categorías únicas


C:\Users\Marin\AppData\Local\Temp\ipykernel_15752\2385515204.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  vars_cat_lgbm  = df_lgbm.select_dtypes(include=['object']).columns.tolist()
C:\Users\Marin\AppData\Local\Temp\ipykernel_15752\2385515204.py:22: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pyda

  WEEKDAY_APPR_PROCESS_START: 7 categorías únicas
  ORGANIZATION_TYPE: 58 categorías únicas

LABEL ENCODING para df_lgbm:
----------------------------------------
  NAME_CONTRACT_TYPE: 2 categorías → números
  CODE_GENDER: 3 categorías → números
  FLAG_OWN_CAR: 2 categorías → números
  FLAG_OWN_REALTY: 2 categorías → números
  NAME_TYPE_SUITE: 8 categorías → números
  NAME_INCOME_TYPE: 8 categorías → números
  NAME_EDUCATION_TYPE: 5 categorías → números
  NAME_FAMILY_STATUS: 6 categorías → números
  NAME_HOUSING_TYPE: 6 categorías → números
  OCCUPATION_TYPE: 19 categorías → números
  WEEKDAY_APPR_PROCESS_START: 7 categorías → números
  ORGANIZATION_TYPE: 58 categorías → números

ONE-HOT ENCODING para df_scorecard:
----------------------------------------
Columnas antes:  67
Columnas después: 168
Columnas nuevas: 101

VERIFICACIÓN:
  df_lgbm categóricas restantes:      0
  df_scorecard categóricas restantes: 0

  df_lgbm shape:      (307511, 67)
  df_scorecard shape: (307511, 168)


In [11]:
# --- CORRECCIÓN Y MEJORA DEL ENCODING ---

# 1. Recargar df_scorecard desde df_lgbm antes del OHE
#    porque ya aplicamos OHE y necesitamos rehacerlo
#    con ORGANIZATION_TYPE agrupado

# Primero reconstruimos df_scorecard desde cero
# usando el df_lgbm como base pero con strings originales

# Necesitamos recargar el dataset original para
# recuperar ORGANIZATION_TYPE como string
# ya que df_lgbm ya tiene label encoding aplicado

print("AGRUPACIÓN DE ORGANIZATION_TYPE")
print("="*60)
print()
print("58 categorías son demasiadas para OHE.")
print("Las agrupamos en 8 grupos con sentido de negocio:")
print()

# Definir grupos basados en tasas de mora del EDA
mapeo_org = {
    # ALTO RIESGO (mora > 10%)
    'Transport: type 3':        'Transporte',
    'Transport: type 4':        'Transporte',
    'Transport: type 2':        'Transporte',
    'Transport: type 1':        'Transporte',
    'Construction':             'Construccion',
    'Industry: type 1':         'Industria',
    'Industry: type 3':         'Industria',
    'Industry: type 4':         'Industria',
    'Industry: type 8':         'Industria',
    'Industry: type 13':        'Industria',
    'Restaurant':               'Servicios_Alto_Riesgo',
    'Cleaning':                 'Servicios_Alto_Riesgo',
    'Agriculture':              'Servicios_Alto_Riesgo',
    'Self-employed':            'Independiente',
    'Realtor':                  'Servicios_Alto_Riesgo',

    # RIESGO MEDIO (mora 7-10%)
    'Business Entity Type 3':   'Empresa_Privada',
    'Business Entity Type 2':   'Empresa_Privada',
    'Business Entity Type 1':   'Empresa_Privada',
    'Trade: type 7':            'Comercio',
    'Trade: type 3':            'Comercio',
    'Trade: type 1':            'Comercio',
    'Trade: type 2':            'Comercio',
    'Security':                 'Seguridad',
    'Mobile':                   'Servicios_Medio_Riesgo',
    'Advertising':              'Servicios_Medio_Riesgo',
    'Postal':                   'Servicios_Medio_Riesgo',
    'Housing':                  'Servicios_Medio_Riesgo',
    'Industry: type 2':         'Industria',
    'Industry: type 5':         'Industria',
    'Industry: type 6':         'Industria',
    'Industry: type 7':         'Industria',
    'Industry: type 9':         'Industria',
    'Industry: type 10':        'Industria',
    'Industry: type 11':        'Industria',
    'Other':                    'Servicios_Medio_Riesgo',
    'Telecom':                  'Servicios_Medio_Riesgo',
    'Legal Services':           'Servicios_Medio_Riesgo',
    'Emergency':                'Servicios_Medio_Riesgo',
    'Electricity':              'Servicios_Medio_Riesgo',
    'Services':                 'Servicios_Medio_Riesgo',
    'Hotel':                    'Servicios_Medio_Riesgo',

    # BAJO RIESGO (mora < 7%)
    'Government':               'Gobierno_Publico',
    'Military':                 'Gobierno_Publico',
    'Police':                   'Gobierno_Publico',
    'Security Ministries':      'Gobierno_Publico',
    'School':                   'Educacion_Salud',
    'University':               'Educacion_Salud',
    'Kindergarten':             'Educacion_Salud',
    'Medicine':                 'Educacion_Salud',
    'Religion':                 'Educacion_Salud',
    'Culture':                  'Educacion_Salud',
    'Bank':                     'Banca_Seguros',
    'Insurance':                'Banca_Seguros',
    'Industry: type 12':        'Industria',
    'Trade: type 4':            'Comercio',
    'Trade: type 5':            'Comercio',
    'Trade: type 6':            'Comercio',
    'HR staff':                 'Servicios_Medio_Riesgo',
    'XNA':                      'Pensionado_NA',
}

# Aplicar agrupación al df original
# Necesitamos recargar el dataset para tener strings
with zipfile.ZipFile(RUTA_ZIP, 'r') as z:
    with z.open('application_train.csv') as f:
        df_org_type = pd.read_csv(f, usecols=['SK_ID_CURR',
                                               'ORGANIZATION_TYPE'])

# Mapear a grupos
df_org_type['ORG_TYPE_GRUPO'] = (
    df_org_type['ORGANIZATION_TYPE']
    .map(mapeo_org)
    .fillna('Otros'))

# Verificar grupos y sus tasas de mora
org_grupos = df_org_type.merge(
    pd.DataFrame({'SK_ID_CURR': df['SK_ID_CURR'],
                  'TARGET': TARGET}),
    on='SK_ID_CURR'
)

print("GRUPOS CREADOS Y TASA DE MORA:")
mora_grupos = (org_grupos.groupby('ORG_TYPE_GRUPO')['TARGET']
               .agg(['mean', 'count'])
               .rename(columns={'mean': 'mora', 'count': 'n'})
               .sort_values('mora', ascending=False))
mora_grupos['mora_pct'] = mora_grupos['mora'] * 100
print(mora_grupos[['n', 'mora_pct']].to_string(
    float_format='{:.2f}'.format))

AGRUPACIÓN DE ORGANIZATION_TYPE

58 categorías son demasiadas para OHE.
Las agrupamos en 8 grupos con sentido de negocio:

GRUPOS CREADOS Y TASA DE MORA:
                            n  mora_pct
ORG_TYPE_GRUPO                         
Construccion             6721     11.68
Servicios_Alto_Riesgo    4921     10.97
Independiente           38412     10.17
Seguridad                3247      9.98
Transporte               8990      9.67
Empresa_Privada         84529      9.12
Comercio                14315      9.07
Industria               14311      8.60
Servicios_Medio_Riesgo  27477      7.62
Educacion_Salud         28757      6.39
Gobierno_Publico        17353      6.19
Pensionado_NA           55374      5.40
Banca_Seguros            3104      5.28


In [12]:
# --- INCORPORAR ORG_TYPE_GRUPO Y REHACER OHE ---

# Agregar el grupo al df_lgbm (ya tiene label encoding)
# Para lgbm: label encoding del grupo
grupos_ordenados = (mora_grupos.sort_values('mora_pct', ascending=False)
                    .index.tolist())

mapeo_grupo_num = {grupo: i for i, grupo in enumerate(grupos_ordenados)}

df_lgbm['ORG_TYPE_GRUPO'] = (df_org_type['ORG_TYPE_GRUPO']
                              .map(mapeo_grupo_num)
                              .values)

print("ORG_TYPE_GRUPO en df_lgbm:")
print(f"  Valores únicos: {df_lgbm['ORG_TYPE_GRUPO'].nunique()}")
print(f"  Rango: {df_lgbm['ORG_TYPE_GRUPO'].min()} - "
      f"{df_lgbm['ORG_TYPE_GRUPO'].max()}")

# Eliminar ORGANIZATION_TYPE original de df_lgbm
# ya está capturada en ORG_TYPE_GRUPO
df_lgbm.drop(columns=['ORGANIZATION_TYPE'], inplace=True)

print(f"\ndf_lgbm shape: {df_lgbm.shape}")

# Para df_scorecard: rehacer desde df_lgbm
# pero necesitamos reconstruir con strings para OHE
# Recargamos df_scorecard desde cero con las
# transformaciones correctas

# Recargar datos originales de variables categóricas
with zipfile.ZipFile(RUTA_ZIP, 'r') as z:
    with z.open('application_train.csv') as f:
        df_cats_original = pd.read_csv(f, usecols=[
            'SK_ID_CURR',
            'NAME_CONTRACT_TYPE', 'CODE_GENDER',
            'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
            'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE',
            'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
            'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE',
            'WEEKDAY_APPR_PROCESS_START'
        ])

# Imputar nulos en categóricas
df_cats_original['NAME_TYPE_SUITE'] = (
    df_cats_original['NAME_TYPE_SUITE'].fillna('Unaccompanied'))
df_cats_original['OCCUPATION_TYPE'] = (
    df_cats_original['OCCUPATION_TYPE'].fillna('Unknown'))

# Agregar grupo de organización
df_cats_original = df_cats_original.merge(
    df_org_type[['SK_ID_CURR', 'ORG_TYPE_GRUPO']],
    on='SK_ID_CURR', how='left')

# Construir df_scorecard combinando numéricas + categóricas OHE
# Primero tomamos las columnas numéricas de df_lgbm
# (sin las categóricas que ya están encoded)
cols_numericas = [c for c in df_lgbm.columns
                  if c not in mapeo_categorias.keys()
                  and c != 'SK_ID_CURR']

# Reconstruir df_scorecard con OHE correcto
df_score_num = df_lgbm[['SK_ID_CURR'] + cols_numericas].copy()

# Columnas categóricas para OHE (sin ORGANIZATION_TYPE original)
cols_ohe = ['NAME_CONTRACT_TYPE', 'CODE_GENDER',
            'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
            'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE',
            'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
            'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE',
            'WEEKDAY_APPR_PROCESS_START', 'ORG_TYPE_GRUPO']

# Aplicar OHE
cats_ohe = pd.get_dummies(
    df_cats_original[cols_ohe],
    drop_first=True,
    dtype=int
)

# Combinar
df_scorecard = pd.concat([df_score_num.reset_index(drop=True),
                          cats_ohe.reset_index(drop=True)],
                         axis=1)

print(f"\ndf_scorecard shape: {df_scorecard.shape}")
print(f"Columnas OHE generadas: {cats_ohe.shape[1]}")
print(f"\nVerificación categóricas restantes: "
      f"{df_scorecard.select_dtypes(include=['str']).shape[1]}")

ORG_TYPE_GRUPO en df_lgbm:
  Valores únicos: 13
  Rango: 0 - 12

df_lgbm shape: (307511, 67)

df_scorecard shape: (307511, 124)
Columnas OHE generadas: 68

Verificación categóricas restantes: 0


In [17]:
# --- PASO 6 CORREGIDO: GUARDAR EN CSV ---

# Usamos CSV en lugar de Parquet por compatibilidad
# CSV es más universal aunque ocupa más espacio

print("PASO 6: GUARDANDO DATASETS PROCESADOS")
print("="*60)

# Agregar TARGET a ambos datasets
df_lgbm_final      = df_lgbm.copy()
df_scorecard_final = df_scorecard.copy()

df_lgbm_final['TARGET']      = TARGET.values
df_scorecard_final['TARGET'] = TARGET.values

print(f"TARGET en df_lgbm:      "
      f"{df_lgbm_final['TARGET'].mean()*100:.2f}% mora ✅")
print(f"TARGET en df_scorecard: "
      f"{df_scorecard_final['TARGET'].mean()*100:.2f}% mora ✅")

# Guardar en CSV
ruta_lgbm      = "data/processed/df_lgbm.csv"
ruta_scorecard = "data/processed/df_scorecard.csv"

print("\nGuardando df_lgbm...")
df_lgbm_final.to_csv(ruta_lgbm, index=False)

print("Guardando df_scorecard...")
df_scorecard_final.to_csv(ruta_scorecard, index=False)

# Verificar tamaños
size_lgbm      = os.path.getsize(ruta_lgbm) / 1024 / 1024
size_scorecard = os.path.getsize(ruta_scorecard) / 1024 / 1024

print(f"\nArchivos guardados:")
print(f"  {ruta_lgbm}")
print(f"  Tamaño: {size_lgbm:.1f} MB")
print(f"  Shape:  {df_lgbm_final.shape}")
print()
print(f"  {ruta_scorecard}")
print(f"  Tamaño: {size_scorecard:.1f} MB")
print(f"  Shape:  {df_scorecard_final.shape}")

# Verificar lectura
df_test = pd.read_csv(ruta_lgbm, nrows=5)
print(f"\nVerificación lectura: ✅")
print(f"  Primeras 5 filas cargadas correctamente")
print(f"  Columnas: {df_test.shape[1]}")

PASO 6: GUARDANDO DATASETS PROCESADOS
TARGET en df_lgbm:      8.07% mora ✅
TARGET en df_scorecard: 8.07% mora ✅

Guardando df_lgbm...
Guardando df_scorecard...

Archivos guardados:
  data/processed/df_lgbm.csv
  Tamaño: 109.9 MB
  Shape:  (307511, 67)

  data/processed/df_scorecard.csv
  Tamaño: 143.2 MB
  Shape:  (307511, 124)

Verificación lectura: ✅
  Primeras 5 filas cargadas correctamente
  Columnas: 67
